# Détection d'anomalies routières — Étape 2 : Prétraitement et organisation des données

**Notebook :** `02_pretraitement.ipynb`

L'étape 1 a montré que le dataset est petit (60 images, 50 sources indépendantes), déséquilibré
(9,85 % de pixels anomaux) et contient des images de même source. Cette étape prépare des données
**prêtes pour l'entraînement**, en appliquant directement ces constats.

| § | Section | Constat de l'étape 1 appliqué |
|---|---|---|
| 0 | Chargement et bibliothèques | — |
| 1 | Fonctions de base | masque binaire : `labels_semantic == 2` |
| 2 | Découpage en 5 plis | catégories déséquilibrées, images de même source |
| 3 | Redimensionnement et normalisation | taille des objets, encodeurs pré-entraînés |
| 4 | Augmentation de données | petit dataset, a priori de position, déséquilibre, variations d'éclairage |
| 5 | `Dataset` et `DataLoader` PyTorch | — |
| 6 | Vérifications | — |
| 7 | Module réutilisable `roadanomaly_lib.py` | — |

**Résultat de l'étape :** `splits.csv` (pli de chaque image) et `roadanomaly_lib.py` (tout le code
de chargement), réutilisés tels quels aux étapes 3 à 5.

## 0. Chargement des données et bibliothèques

Comme à l'étape 1 : exécutez la cellule, cliquez sur **« Choisir les fichiers »** et sélectionnez
`RoadAnomaly.zip`.

In [ ]:
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def trouver_dataset(dossier):
    dossier = Path(dossier)
    if not dossier.exists():
        return None
    hits = sorted(dossier.rglob("frame_list.json"))
    return hits[0].parent if hits else None

if IN_COLAB:
    BASE = Path("/content")
    DATA_ROOT = trouver_dataset(BASE / "data")
    if DATA_ROOT is None:
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip :")
        for nom in files.upload():
            with zipfile.ZipFile(BASE / nom) as z:
                z.extractall(BASE / "data")
            os.remove(BASE / nom)
        DATA_ROOT = trouver_dataset(BASE / "data")
else:
    BASE = Path.cwd()
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : le zip doit contenir frame_list.json et frames/"
OUT = BASE / "outputs_etape2"; FIG_DIR = OUT / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset :", DATA_ROOT)

Les bibliothèques **PyTorch** (réseaux de neurones) et **albumentations** (augmentation d'images
et de masques) sont préinstallées sur Colab. La cellule vérifie qu'albumentations est en version 2,
celle utilisée par ce notebook, et l'installe sinon.

In [ ]:
import importlib
try:
    import albumentations
    ok = int(albumentations.__version__.split(".")[0]) >= 2
except ImportError:
    ok = False
if not ok:
    %pip install -q "albumentations==2.0.8"
    importlib.invalidate_caches()

import albumentations, torch
print("albumentations", albumentations.__version__, "| torch", torch.__version__,
      "| GPU :", torch.cuda.is_available())

## 1. Fonctions de base

Mêmes fonctions qu'à l'étape 1. Le point clé du prétraitement des masques est dans `load_mask` :
le masque brut contient 0 (fond) et 2 (anomalie) ; on le convertit en **masque binaire {0, 1}**,
le format attendu par une sortie sigmoïde et par les fonctions de perte (BCE, Dice).

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image

ANOMALY_LABEL = 2
CATEGORIES = ["animals", "obstacles", "cones", "lost_tires", "vehicle"]

# Images issues de la même source (cf. étape 1, §4.3) : jamais réparties entre entraînement et test
SOURCE_GROUPS = {
    "youtube_lost_tires": ["lost_tires03", "lost_tires08", "lost_tires09"],
    "cardboard_hedgehogs": ["obstacles03_cardboard_czech_hedgehog", "obstacles04_cardboard_czech_hedgehog",
                            "obstacles05_cardboard_czech_hedgehog", "obstacles06_cardboard_czech_hedgehog"],
    "tripadvisor_rocks": ["obstacles12_rocks2", "obstacles12_rocks4", "obstacles12_rocks5", "obstacles12_rocks6"],
    "papakura_news": ["cones07", "cones08_can"],
    "broadmoor": ["cones02_Broadmoor_Manhole_Cones", "vehicle02_Broadmoor_Cones_Skidloader"],
}
_GROUPE = {n: g for g, noms in SOURCE_GROUPS.items() for n in noms}

def list_frames(root):
    with open(Path(root) / "frame_list.json") as f:
        return [Path(x).stem for x in json.load(f)]

def category_of(name):
    m = re.match(r"([a-z_]+?)\d", name)
    return m.group(1) if m else name.split("_")[0]

def source_group(name):
    return _GROUPE.get(name, name)

def load_image(root, name):
    """Image RGB uint8 (720, 1280, 3)."""
    return np.array(Image.open(Path(root) / "frames" / f"{name}.webp").convert("RGB"))

def load_mask(root, name):
    """Masque binaire uint8 (720, 1280) : 1 = anomalie, 0 = fond."""
    sem = np.array(Image.open(Path(root) / "frames" / f"{name}.labels" / "labels_semantic.png"))
    return (sem == ANOMALY_LABEL).astype(np.uint8)

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False})
CAT_COLORS = {"animals": "#2a78d6", "obstacles": "#eb6834", "cones": "#1baf7a",
              "lost_tires": "#eda100", "vehicle": "#e87ba4"}
CAT_FR = {"animals": "animaux", "obstacles": "obstacles", "cones": "cônes",
          "lost_tires": "pneus perdus", "vehicle": "véhicules"}

frames = list_frames(DATA_ROOT)
m = load_mask(DATA_ROOT, frames[0])
print(f"{len(frames)} images | masque : forme {m.shape}, type {m.dtype}, valeurs {np.unique(m)}")

## 2. Découpage des données : validation croisée en 5 plis

### Pourquoi pas un simple découpage 70 / 15 / 15 ?

Avec 60 images, un test de 9 images donne un score qui dépend surtout du hasard du découpage.
On utilise donc une **validation croisée en 5 plis** : les images sont réparties en 5 plis
d'environ 12 images, et on fait **5 tours**. Au tour *k* :

| Rôle | Plis | Part |
|---|---|---|
| **test** | pli *k* | ≈ 20 % |
| **validation** (choix de l'époque, du seuil) | pli *k + 1* | ≈ 20 % |
| **entraînement** | les 3 autres | ≈ 60 % |

Chaque image sert exactement une fois en test. On rapporte la **moyenne ± l'écart-type** des 5
tours.

### Deux contraintes, une seule fonction

- **Stratification par catégorie** : chaque pli garde à peu près les proportions de catégories.
- **Groupement par source** : toutes les images d'une même source vont dans le **même pli**, pour
  éviter la fuite de données (étape 1, §4.3).

`StratifiedGroupKFold` (scikit-learn) respecte les deux. La graine `SEED = 42` rend le découpage
**reproductible** : il sera identique aux étapes suivantes.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS, SEED = 5, 42

def make_splits(root, n_folds=N_FOLDS, seed=SEED):
    """Attribue chaque image à un pli (0..n_folds-1).
    Stratifié par catégorie, groupé par source : une source n'apparaît que dans un seul pli."""
    names = list_frames(root)
    df = pd.DataFrame({"name": names,
                       "category": [category_of(n) for n in names],
                       "group": [source_group(n) for n in names]})
    df["fold"] = -1
    import warnings   # avertissement attendu : la classe la plus rare (pneus) n'a que 3 images
    warnings.filterwarnings("ignore", message="The least populated class")
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(sgkf.split(df, y=df["category"], groups=df["group"])):
        df.loc[test_idx, "fold"] = k
    return df

def fold_sets(splits, k, n_folds=N_FOLDS):
    """Tour k de la validation croisée : test = pli k, validation = pli k+1, entraînement = les 3 autres."""
    val_fold = (k + 1) % n_folds
    test  = splits.loc[splits.fold == k, "name"].tolist()
    val   = splits.loc[splits.fold == val_fold, "name"].tolist()
    train = splits.loc[~splits.fold.isin([k, val_fold]), "name"].tolist()
    return train, val, test

splits = make_splits(DATA_ROOT)
splits.head()

### Vérifications du découpage

In [ ]:
# 1) aucune source dans deux plis (pas de fuite de données)
plis_par_groupe = splits.groupby("group")["fold"].nunique()
assert (plis_par_groupe == 1).all(), "une source est répartie sur plusieurs plis !"
print("Aucune source n'est répartie sur deux plis.")

# 2) chaque image dans exactement un pli
assert splits["fold"].between(0, N_FOLDS - 1).all() and splits["name"].is_unique

# 3) composition des plis
compo = pd.crosstab(splits["fold"], splits["category"]).reindex(columns=CATEGORIES, fill_value=0)
compo["total"] = compo.sum(axis=1)
compo.index = [f"pli {k}" for k in compo.index]
display(compo)

# 4) les 5 tours : tailles train / val / test, et aucune image en commun
for k in range(N_FOLDS):
    tr, va, te = fold_sets(splits, k)
    assert not (set(tr) & set(va) or set(tr) & set(te) or set(va) & set(te))
    print(f"tour {k} : entraînement {len(tr):2d} | validation {len(va):2d} | test {len(te):2d}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
gauche = np.zeros(N_FOLDS)
for c in CATEGORIES:
    v = pd.crosstab(splits.fold, splits.category).reindex(columns=CATEGORIES, fill_value=0)[c].values
    ax.barh([f"pli {k}" for k in range(N_FOLDS)], v, left=gauche, color=CAT_COLORS[c],
            edgecolor="white", linewidth=1.5, label=CAT_FR[c], height=0.6)
    gauche += v
ax.invert_yaxis(); ax.set_xlabel("Nombre d'images")
ax.set_title("Composition des 5 plis par catégorie", fontweight="bold")
ax.legend(frameon=False, ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ax.grid(axis="x", color="#e4e3df"); ax.set_axisbelow(True)
plt.tight_layout(); fig.savefig(FIG_DIR / "01_composition_plis.png"); plt.show()

print("Pli contenant les 3 images de pneus (une seule vidéo) :",
      splits.loc[splits.category == "lost_tires", "fold"].unique())

**À retenir**

- Les plis contiennent 11 à 13 images avec des proportions de catégories proches de celles du
  dataset. La stratification ne peut pas être parfaite : les groupes de source doivent rester
  entiers.
- **Les 3 images de pneus sont dans un seul pli** (une seule vidéo). Au tour où ce pli est en
  test, le modèle n'a vu aucun pneu à l'entraînement : un vrai test sur un type d'anomalie
  inconnu. Au tour où il est en validation, aucun pneu n'est en test.
- **Coût en calcul :** 5 tours × 5 modèles = 25 entraînements. En pratique, on règle les
  hyperparamètres sur le tour 0, puis on lance les 5 tours avec les réglages retenus.

## 3. Redimensionnement et normalisation

### Résolution de travail : 960 × 540

L'étape 1 (§7.1) a montré que réduire la résolution fait disparaître les petits objets :

| Résolution | Objets < 64 px |
|---|---|
| 1280 × 720 | 0 % |
| **960 × 540** | **1 %** |
| 640 × 360 | 18 % |
| 512 × 288 | 21 % |

**960 × 540** (75 % de l'original) est le bon compromis : presque aucun objet perdu, et 44 % de
pixels en moins qu'en pleine résolution. Les images sont redimensionnées **une seule fois** au
chargement :

- images : interpolation `INTER_AREA` (moyenne des pixels, la meilleure pour réduire) ;
- masques : interpolation `INTER_NEAREST` (plus proche voisin), qui garde des valeurs **0 ou 1
  exactement** ; une interpolation linéaire créerait des valeurs intermédiaires (0,5…) sans sens.

### Entraînement sur découpes, évaluation sur l'image entière

- **Entraînement :** découpes de **512 × 512**. Un lot de 4 découpes tient dans la mémoire d'un
  GPU T4, et chaque époque voit des parties différentes des images.
- **Validation et test :** **image entière** 960 × 540. Les réseaux de segmentation étant
  entièrement convolutifs, ils acceptent une taille différente. L'image est complétée par des
  zéros en bas jusqu'à **544** lignes, car U-Net et DeepLabV3+ divisent la taille par 2 cinq fois
  (2⁵ = 32) et 540 n'est pas un multiple de 32.

### Normalisation ImageNet

Chaque canal est normalisé avec la moyenne et l'écart-type d'**ImageNet**
(moyenne 0,485 / 0,456 / 0,406, écart-type 0,229 / 0,224 / 0,225). Les encodeurs pré-entraînés
(ResNet34…) ont appris sur des images normalisées ainsi ; leur fournir la même distribution est
indispensable au transfer learning.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

WORK_W, WORK_H = 960, 540        # résolution de travail (75 % de 1280 x 720)
CROP = 512                       # taille des découpes d'entraînement
MEAN = (0.485, 0.456, 0.406)     # statistiques ImageNet (encodeurs pré-entraînés)
STD  = (0.229, 0.224, 0.225)

def get_train_transform(crop=CROP, p_anomaly_crop=0.5):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.8, 1.25), rotate=(-5, 5), border_mode=cv2.BORDER_CONSTANT, p=0.7),
        A.PadIfNeeded(min_height=crop, min_width=crop, border_mode=cv2.BORDER_CONSTANT),
        A.OneOf([   # moitié des découpes centrées sur une anomalie, moitié au hasard
            A.CropNonEmptyMaskIfExists(height=crop, width=crop, p=p_anomaly_crop),
            A.RandomCrop(height=crop, width=crop, p=1 - p_anomaly_crop),
        ], p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.OneOf([A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=(3, 5))], p=0.2),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_eval_transform():
    # image entière ; complétée en bas/à droite jusqu'à un multiple de 32 (540 -> 544)
    return A.Compose([
        A.PadIfNeeded(min_height=None, min_width=None, pad_height_divisor=32, pad_width_divisor=32,
                      position="top_left", border_mode=cv2.BORDER_CONSTANT),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def denormalize(t):
    """Tenseur (3, H, W) normalisé -> image uint8 (H, W, 3) pour l'affichage."""
    x = t.detach().cpu().numpy().transpose(1, 2, 0) * np.array(STD) + np.array(MEAN)
    return (x.clip(0, 1) * 255).astype(np.uint8)

print(f"Résolution de travail : {WORK_W} x {WORK_H} | découpes : {CROP} x {CROP}")

## 4. Augmentation de données

Avec 35 à 37 images d'entraînement par tour, le modèle mémoriserait vite les images
(**sur-apprentissage**). L'augmentation crée à chaque époque des variantes réalistes. Chaque
transformation répond à un constat de l'étape 1 :

| Transformation | Paramètres | Justification |
|---|---|---|
| Retournement horizontal | p = 0,5 | une scène retournée reste réaliste ; limite le raccourci « position = anomalie » |
| Affine : échelle, rotation | ×0,8 à ×1,25 ; ±5° ; p = 0,7 | objets de toutes tailles (100 à 300 000 px) ; légère inclinaison de la caméra |
| Découpe 512 × 512 | 50 % centrée sur une anomalie, 50 % au hasard | garantit qu'une découpe contient l'anomalie, même petite ; effet mesuré au §6.3 |
| Luminosité, contraste | ±25 % ; p = 0,5 | luminosité 60 à 163 et contraste 30 à 84 selon les images |
| Teinte, saturation | p = 0,3 | caméras et éclairages différents |
| Flou (mouvement ou gaussien) | p = 0,2 | captures de caméra embarquée, véhicule en mouvement |

**Ce qu'on évite :** le retournement vertical (une route à l'envers n'existe pas) et les
rotations fortes (l'horizon reste à peu près horizontal).

**Albumentations applique la même transformation géométrique à l'image et au masque** :
le masque reste aligné pixel par pixel. Les transformations de couleur ne touchent que l'image.

In [ ]:
# Une même image, 8 tirages aléatoires du pipeline d'entraînement
n = "cones03_Bang_Na_Trat_Highway"
img = cv2.resize(load_image(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_AREA)
msk = cv2.resize(load_mask(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_NEAREST)
tf = get_train_transform()

fig, axes = plt.subplots(2, 5, figsize=(16, 6.6), gridspec_kw={"width_ratios": [1.9, 1, 1, 1, 1]})
gs = axes[0, 0].get_gridspec()
for ax in axes[:, 0]: ax.remove()
ax0 = fig.add_subplot(gs[:, 0])
ax0.imshow(img); ax0.set_title(f"Original ({WORK_W}×{WORK_H})"); ax0.axis("off")
for ax in axes[:, 1:].ravel():
    out = tf(image=img, mask=msk)
    vis = denormalize(out["image"])
    m = out["mask"].numpy().astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ax.imshow(cv2.drawContours(vis.copy(), contours, -1, (0, 200, 255), 3))
    ax.set_title(f"{m.mean():.1%} anomalie", fontsize=9); ax.axis("off")
fig.suptitle("Découpes augmentées 512×512 (contour cyan = masque transformé avec l'image)", fontweight="bold")
plt.tight_layout(); fig.savefig(FIG_DIR / "02_exemples_augmentation.png"); plt.show()

## 5. `Dataset` et `DataLoader` PyTorch

- **`RoadAnomalyDataset`** charge les images d'une liste, les redimensionne une fois et les garde
  en mémoire (60 images ≈ 90 Mo : relire les fichiers à chaque époque serait inutilement lent).
  Chaque accès applique la transformation et renvoie un dictionnaire :
  `image` (3, H, W) normalisée, `mask` (1, H, W) en 0/1, `name`.
- **`samples_per_image = 4`** : chaque image d'entraînement donne 4 découpes différentes par
  époque (tour 0 : 37 images → 148 découpes, 37 lots de 4).
- **`make_loaders(root, splits, k)`** renvoie les trois `DataLoader` du tour *k* :
  entraînement mélangé (`shuffle=True`), validation et test image par image.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class RoadAnomalyDataset(Dataset):
    """Images et masques redimensionnés une fois à WORK_W x WORK_H et gardés en mémoire.
    samples_per_image > 1 : plusieurs découpes aléatoires de chaque image par époque."""

    def __init__(self, root, names, transform, samples_per_image=1):
        self.names, self.transform, self.k = list(names), transform, samples_per_image
        self.images, self.masks = {}, {}
        for n in self.names:
            img = cv2.resize(load_image(root, n), (WORK_W, WORK_H), interpolation=cv2.INTER_AREA)
            msk = cv2.resize(load_mask(root, n), (WORK_W, WORK_H), interpolation=cv2.INTER_NEAREST)
            self.images[n], self.masks[n] = img, msk

    def __len__(self):
        return len(self.names) * self.k

    def __getitem__(self, i):
        name = self.names[i % len(self.names)]
        out = self.transform(image=self.images[name], mask=self.masks[name])
        mask = out["mask"].float().unsqueeze(0)          # (1, H, W), valeurs 0 / 1
        return {"image": out["image"], "mask": mask, "name": name}

def make_loaders(root, splits, k, batch_size=4, samples_per_image=4, num_workers=2):
    """DataLoaders du tour k : entraînement (découpes augmentées), validation et test (images entières)."""
    train, val, test = fold_sets(splits, k)
    ds_train = RoadAnomalyDataset(root, train, get_train_transform(), samples_per_image)
    ds_val   = RoadAnomalyDataset(root, val, get_eval_transform())
    ds_test  = RoadAnomalyDataset(root, test, get_eval_transform())
    return (DataLoader(ds_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True),
            DataLoader(ds_val, batch_size=1, shuffle=False, num_workers=num_workers),
            DataLoader(ds_test, batch_size=1, shuffle=False, num_workers=num_workers))

train_loader, val_loader, test_loader = make_loaders(DATA_ROOT, splits, k=0)
print(f"Tour 0 : {len(train_loader.dataset)} découpes d'entraînement par époque "
      f"({len(train_loader)} lots), {len(val_loader.dataset)} images de validation, "
      f"{len(test_loader.dataset)} images de test")

## 6. Vérifications

Un bug de prétraitement (masque décalé, mauvaise normalisation) ne provoque pas d'erreur : le
modèle apprend simplement mal. On vérifie donc tout avant d'entraîner.

### 6.1 Un lot d'entraînement

In [ ]:
batch = next(iter(train_loader))
x, y = batch["image"], batch["mask"]
print("image :", tuple(x.shape), x.dtype, f"| valeurs de {x.min():.2f} à {x.max():.2f} (normalisation ImageNet : environ -2,1 à +2,6)")
print("masque:", tuple(y.shape), y.dtype, "| valeurs :", torch.unique(y).tolist())
assert x.shape[1:] == (3, CROP, CROP) and y.shape[1:] == (1, CROP, CROP)
assert set(torch.unique(y).tolist()) <= {0.0, 1.0}

fig, axes = plt.subplots(2, len(x), figsize=(3.2 * len(x), 6.4))
for i in range(len(x)):
    axes[0, i].imshow(denormalize(x[i])); axes[0, i].set_title(batch["name"][i][:22], fontsize=8)
    axes[1, i].imshow(y[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"masque : {y[i].mean():.1%} anomalie", fontsize=8)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); fig.savefig(FIG_DIR / "03_lot_entrainement.png"); plt.show()

### 6.2 Validation et test : image entière

In [ ]:
b = next(iter(val_loader))
print("image :", tuple(b["image"].shape), "| masque :", tuple(b["mask"].shape))
assert b["image"].shape[-2:] == (544, 960), "540 lignes + 4 lignes de remplissage attendues"
assert b["mask"][..., WORK_H:, :].sum() == 0, "la zone de remplissage ne doit contenir aucune anomalie"
print(f"OK : {WORK_H} lignes d'image + {544 - WORK_H} lignes de remplissage (exclues lors de l'évaluation)")

### 6.3 Effet de la découpe orientée vers les anomalies

On compare, sur 400 découpes d'entraînement, la part de pixels anomaux et la part de découpes
**sans aucune anomalie** : découpes 100 % au hasard contre notre stratégie (50 % centrées sur une
anomalie).

In [ ]:
train_names, _, _ = fold_sets(splits, 0)
ds_rand = RoadAnomalyDataset(DATA_ROOT, train_names, get_train_transform(p_anomaly_crop=0.0))
ds_ours = RoadAnomalyDataset(DATA_ROOT, train_names, get_train_transform(p_anomaly_crop=0.5))

def stats(ds, n=400, seed=0):
    rng = np.random.default_rng(seed)
    fr = np.array([ds[int(i)]["mask"].mean().item() for i in rng.integers(0, len(ds), n)])
    return fr.mean(), (fr == 0).mean()

res = pd.DataFrame([("100 % au hasard", *stats(ds_rand)), ("50 % centrées sur une anomalie", *stats(ds_ours))],
                   columns=["stratégie de découpe", "pixels anomaux (moyenne)", "découpes sans anomalie"])
display(res.style.format({"pixels anomaux (moyenne)": "{:.1%}", "découpes sans anomalie": "{:.0%}"}))

**Lecture :** l'effet est faible, et c'est une information utile. Les deux stratégies donnent à
peu près la même part de pixels anomaux (environ 15 %) et presque aucune découpe vide. La raison :
à 960 × 540, une découpe de 512 × 512 couvre environ la moitié de l'image, et les anomalies sont
concentrées au centre (étape 1, §8). Une découpe au hasard contient donc presque toujours une
anomalie.

**Conséquences :**

- On garde la découpe orientée (50 %) : elle ne coûte rien et garantit qu'un petit objet au bord
  de l'image est vu de temps en temps.
- Les découpes contiennent plus d'anomalies (≈ 15 %) que les images entières (9,85 %), car elles
  coupent les bords, rarement anomaux. Le modèle verra donc un peu plus de positifs à
  l'entraînement qu'au test : un léger écart à garder en tête pour le choix du seuil, réglé sur
  la validation en images entières.
- **Le déséquilibre n'est pas résolu par la découpe** : il reste environ 85 % de pixels de fond.
  Il sera traité par la fonction de perte (Dice + BCE) à l'étape 3.

## 7. Module réutilisable et sauvegarde

Tout le code de chargement est regroupé dans **`roadanomaly_lib.py`**. Aux étapes 3 à 5, il
suffira d'envoyer ce fichier avec `RoadAnomaly.zip` puis d'écrire
`from roadanomaly_lib import *` : chaque modèle utilisera exactement le même découpage et le même
prétraitement, condition d'une comparaison équitable.

In [ ]:
%%writefile roadanomaly_lib.py
"""roadanomaly_lib.py — chargement, découpage et prétraitement du dataset RoadAnomaly.
Généré par 02_pretraitement.ipynb ; réutilisé par les étapes 3 à 5."""

import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image

ANOMALY_LABEL = 2
CATEGORIES = ["animals", "obstacles", "cones", "lost_tires", "vehicle"]

# Images issues de la même source (cf. étape 1, §4.3) : jamais réparties entre entraînement et test
SOURCE_GROUPS = {
    "youtube_lost_tires": ["lost_tires03", "lost_tires08", "lost_tires09"],
    "cardboard_hedgehogs": ["obstacles03_cardboard_czech_hedgehog", "obstacles04_cardboard_czech_hedgehog",
                            "obstacles05_cardboard_czech_hedgehog", "obstacles06_cardboard_czech_hedgehog"],
    "tripadvisor_rocks": ["obstacles12_rocks2", "obstacles12_rocks4", "obstacles12_rocks5", "obstacles12_rocks6"],
    "papakura_news": ["cones07", "cones08_can"],
    "broadmoor": ["cones02_Broadmoor_Manhole_Cones", "vehicle02_Broadmoor_Cones_Skidloader"],
}
_GROUPE = {n: g for g, noms in SOURCE_GROUPS.items() for n in noms}

def list_frames(root):
    with open(Path(root) / "frame_list.json") as f:
        return [Path(x).stem for x in json.load(f)]

def category_of(name):
    m = re.match(r"([a-z_]+?)\d", name)
    return m.group(1) if m else name.split("_")[0]

def source_group(name):
    return _GROUPE.get(name, name)

def load_image(root, name):
    """Image RGB uint8 (720, 1280, 3)."""
    return np.array(Image.open(Path(root) / "frames" / f"{name}.webp").convert("RGB"))

def load_mask(root, name):
    """Masque binaire uint8 (720, 1280) : 1 = anomalie, 0 = fond."""
    sem = np.array(Image.open(Path(root) / "frames" / f"{name}.labels" / "labels_semantic.png"))
    return (sem == ANOMALY_LABEL).astype(np.uint8)


from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS, SEED = 5, 42

def make_splits(root, n_folds=N_FOLDS, seed=SEED):
    """Attribue chaque image à un pli (0..n_folds-1).
    Stratifié par catégorie, groupé par source : une source n'apparaît que dans un seul pli."""
    names = list_frames(root)
    df = pd.DataFrame({"name": names,
                       "category": [category_of(n) for n in names],
                       "group": [source_group(n) for n in names]})
    df["fold"] = -1
    import warnings   # avertissement attendu : la classe la plus rare (pneus) n'a que 3 images
    warnings.filterwarnings("ignore", message="The least populated class")
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(sgkf.split(df, y=df["category"], groups=df["group"])):
        df.loc[test_idx, "fold"] = k
    return df

def fold_sets(splits, k, n_folds=N_FOLDS):
    """Tour k de la validation croisée : test = pli k, validation = pli k+1, entraînement = les 3 autres."""
    val_fold = (k + 1) % n_folds
    test  = splits.loc[splits.fold == k, "name"].tolist()
    val   = splits.loc[splits.fold == val_fold, "name"].tolist()
    train = splits.loc[~splits.fold.isin([k, val_fold]), "name"].tolist()
    return train, val, test


import albumentations as A
from albumentations.pytorch import ToTensorV2

WORK_W, WORK_H = 960, 540        # résolution de travail (75 % de 1280 x 720)
CROP = 512                       # taille des découpes d'entraînement
MEAN = (0.485, 0.456, 0.406)     # statistiques ImageNet (encodeurs pré-entraînés)
STD  = (0.229, 0.224, 0.225)

def get_train_transform(crop=CROP, p_anomaly_crop=0.5):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.8, 1.25), rotate=(-5, 5), border_mode=cv2.BORDER_CONSTANT, p=0.7),
        A.PadIfNeeded(min_height=crop, min_width=crop, border_mode=cv2.BORDER_CONSTANT),
        A.OneOf([   # moitié des découpes centrées sur une anomalie, moitié au hasard
            A.CropNonEmptyMaskIfExists(height=crop, width=crop, p=p_anomaly_crop),
            A.RandomCrop(height=crop, width=crop, p=1 - p_anomaly_crop),
        ], p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.OneOf([A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=(3, 5))], p=0.2),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_eval_transform():
    # image entière ; complétée en bas/à droite jusqu'à un multiple de 32 (540 -> 544)
    return A.Compose([
        A.PadIfNeeded(min_height=None, min_width=None, pad_height_divisor=32, pad_width_divisor=32,
                      position="top_left", border_mode=cv2.BORDER_CONSTANT),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def denormalize(t):
    """Tenseur (3, H, W) normalisé -> image uint8 (H, W, 3) pour l'affichage."""
    x = t.detach().cpu().numpy().transpose(1, 2, 0) * np.array(STD) + np.array(MEAN)
    return (x.clip(0, 1) * 255).astype(np.uint8)


import torch
from torch.utils.data import Dataset, DataLoader

class RoadAnomalyDataset(Dataset):
    """Images et masques redimensionnés une fois à WORK_W x WORK_H et gardés en mémoire.
    samples_per_image > 1 : plusieurs découpes aléatoires de chaque image par époque."""

    def __init__(self, root, names, transform, samples_per_image=1):
        self.names, self.transform, self.k = list(names), transform, samples_per_image
        self.images, self.masks = {}, {}
        for n in self.names:
            img = cv2.resize(load_image(root, n), (WORK_W, WORK_H), interpolation=cv2.INTER_AREA)
            msk = cv2.resize(load_mask(root, n), (WORK_W, WORK_H), interpolation=cv2.INTER_NEAREST)
            self.images[n], self.masks[n] = img, msk

    def __len__(self):
        return len(self.names) * self.k

    def __getitem__(self, i):
        name = self.names[i % len(self.names)]
        out = self.transform(image=self.images[name], mask=self.masks[name])
        mask = out["mask"].float().unsqueeze(0)          # (1, H, W), valeurs 0 / 1
        return {"image": out["image"], "mask": mask, "name": name}

def make_loaders(root, splits, k, batch_size=4, samples_per_image=4, num_workers=2):
    """DataLoaders du tour k : entraînement (découpes augmentées), validation et test (images entières)."""
    train, val, test = fold_sets(splits, k)
    ds_train = RoadAnomalyDataset(root, train, get_train_transform(), samples_per_image)
    ds_val   = RoadAnomalyDataset(root, val, get_eval_transform())
    ds_test  = RoadAnomalyDataset(root, test, get_eval_transform())
    return (DataLoader(ds_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True),
            DataLoader(ds_val, batch_size=1, shuffle=False, num_workers=num_workers),
            DataLoader(ds_test, batch_size=1, shuffle=False, num_workers=num_workers))

In [ ]:
# Vérifie que le module fonctionne seul et redonne le même découpage
import importlib, roadanomaly_lib
importlib.reload(roadanomaly_lib)
assert roadanomaly_lib.make_splits(DATA_ROOT).equals(splits), "découpage différent !"
print("roadanomaly_lib.py : import OK, découpage identique")

splits.to_csv(OUT / "splits.csv", index=False)
import shutil
shutil.copy("roadanomaly_lib.py", OUT / "roadanomaly_lib.py")
print("Fichiers :", *sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file()), sep="\n  ")

if IN_COLAB:
    from google.colab import files
    shutil.make_archive(str(BASE / "resultats_etape2"), "zip", OUT)
    files.download(str(BASE / "resultats_etape2.zip"))

## Synthèse de l'étape 2

| Élément | Choix | Justification (étape 1) |
|---|---|---|
| Masques | binaires {0, 1}, redimensionnés au plus proche voisin | valeurs 0/2 d'origine ; pas de valeurs intermédiaires |
| Découpage | 5 plis, `StratifiedGroupKFold`, graine 42 | 60 images ; catégories déséquilibrées ; 5 groupes de même source |
| Tour *k* | test = pli *k*, validation = pli *k+1*, entraînement = 3 plis | chaque image testée une fois ; moyenne ± écart-type |
| Résolution | 960 × 540 (1 % d'objets < 64 px) | 21 % d'objets perdus à 512 × 288 |
| Entraînement | découpes 512 × 512, 4 par image et par époque | mémoire GPU ; variété |
| Évaluation | image entière, complétée à 544 lignes | réseaux entièrement convolutifs ; multiple de 32 |
| Normalisation | moyenne et écart-type ImageNet | encodeurs pré-entraînés |
| Augmentation | retournement, échelle, découpe orientée, couleur, flou | petit dataset, a priori de position, déséquilibre, éclairage |

**Étape suivante — `03_modeles.ipynb` :** entraînement des modèles, en commençant par le U-Net
de référence et le U-Net pré-entraîné. À garder pour l'étape 3 : `RoadAnomaly.zip` et
`roadanomaly_lib.py` (dans `resultats_etape2.zip`).